# Bankruptcy cascade — p-threshold sweep (Dataset 1/2 methodology)

Self-contained. Same exact cascade model as `bankruptcy_cascade_python_replica.ipynb`, extended with the
**p-threshold sensitivity** used for Dataset 1 and Dataset 2.

**How `p` maps in:** in the equity model a bank defaults after losing its buffer, and the sweep does
`Equity = (1 - p) * Equity`. Here the failure buffer is `Ki` (a bank fails when incoming weight from
failed banks `> Ki`), so the faithful analog is the same shrink applied to `Ki`:

$$Ki(p) = (1 - p)\,Ki_{base}\qquad(\text{and }(1-p)\,Ki_{target}\text{ under target\_policy})$$

`p = 0` reproduces the baseline R model (`Ki = 0.04`); higher `p` ⇒ smaller buffer ⇒ larger cascades.

**Outputs per `p`** (matching your target schema): a node dataset and a target file
`[bank_id, systemic_risk_label, log_systemic_risk_label]` with `systemic_risk_label = cascade_size`,
`log_systemic_risk_label = log1p(cascade_size)`. `p` set = **p0 + p5..p40** (as Dataset 2).

In [10]:
from pathlib import Path
import random
import numpy as np
import pandas as pd

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / 'src').exists() and (p / 'requirements.txt').exists():
            return p
    raise FileNotFoundError('Project root not found.')

PROJECT_ROOT = find_project_root()
R_DIR = PROJECT_ROOT / 'financial-contagion-in-R'
NETWORK_SIZE = 10000
KI_BASE, KI_TARGET = 0.04, 0.10
AVG_DEGREE_LABEL = '1_5'
AVG_DEGREE = 1.5
print('R dir:', R_DIR)

R dir: /Users/rubenmarques/Documents/Repositórios/Thesis/financial-contagion-in-R


## Cascade model (dict/array version — fast) with `Ki(p)` buffer

In [11]:
def build_graph(edges_csv, network_size=NETWORK_SIZE):
    e = pd.read_csv(edges_csv)
    n = max(int(network_size), int(e[['from', 'to']].max().max()))
    out_nb = {i: [] for i in range(1, n + 1)}
    in_w   = {i: {} for i in range(1, n + 1)}
    deg_in  = np.zeros(n + 1); deg_out = np.zeros(n + 1)
    str_in  = np.zeros(n + 1); str_out = np.zeros(n + 1)
    for f, t, w in zip(e['from'].astype(int), e['to'].astype(int), e['weight'].astype(float)):
        out_nb[f].append(t); in_w[t][f] = w
        deg_out[f] += 1; deg_in[t] += 1; str_out[f] += w; str_in[t] += w
    deg_total = deg_in + deg_out
    g = dict(n=n, out_nb=out_nb, in_w=in_w, deg_in=deg_in, deg_out=deg_out,
             deg_total=deg_total, str_in=str_in, str_out=str_out, str_total=str_in + str_out)
    g['deg_q95'] = float(np.quantile(deg_total[1:n + 1], 0.95))
    return g


def judge_bankrupt(g, this_batch_neighbor, already_bankrupt, target_policy=False,
                   ki_base=KI_BASE, ki_target=KI_TARGET):
    in_w, deg_total, q95 = g['in_w'], g['deg_total'], g['deg_q95']
    Ki = ki_base
    bankrupt = []
    for i in this_batch_neighbor:
        if target_policy and deg_total[i] >= q95:
            Ki = ki_target
        s = 0.0; has_bankrupt_in = False
        for j, w in in_w[i].items():
            if j in already_bankrupt:
                s += w; has_bankrupt_in = True
        if has_bankrupt_in and s > Ki:
            bankrupt.append(i)
    return set(bankrupt)


def simulate_bankrupt(g, method='node', type='banks', target_policy=False, initial_node=None,
                      ki_base=KI_BASE, ki_target=KI_TARGET, rng=None):
    out_nb, deg_total, n = g['out_nb'], g['deg_total'], g['n']
    if method == 'biggest':
        initial = int(np.argmax(deg_total[1:n + 1])) + 1
    elif method == 'node':
        initial = int(initial_node)
    else:
        initial = (rng or random).randint(1, n)
    bankrupt = {initial}; this_batch = [initial]; number_bankrupt = 1
    while True:
        cand = set()
        for b in this_batch:
            cand.update(out_nb[b])
        new_b = judge_bankrupt(g, cand, bankrupt, target_policy=target_policy,
                               ki_base=ki_base, ki_target=ki_target)
        bankrupt |= new_b
        if len(bankrupt) == number_bankrupt:
            break
        number_bankrupt = len(bankrupt); this_batch = list(new_b)
    return len(bankrupt) if type == 'num' else sorted(bankrupt)


def compute_node_cascades(edges_csv, avg_degree, network_size=NETWORK_SIZE, target_policy=False, p=0.0):
    g = build_graph(edges_csv, network_size); n = g['n']
    ki_base, ki_target = (1.0 - p) * KI_BASE, (1.0 - p) * KI_TARGET
    cascade = np.ones(n + 1, dtype=int)
    for node in range(1, n + 1):
        cascade[node] = simulate_bankrupt(g, method='node', initial_node=node, type='num',
                                          target_policy=target_policy, ki_base=ki_base, ki_target=ki_target)
    df = pd.DataFrame({
        'avg_degree': avg_degree, 'node_id': np.arange(1, n + 1),
        'degree_in': g['deg_in'][1:n + 1].astype(int), 'degree_out': g['deg_out'][1:n + 1].astype(int),
        'degree_total': g['deg_total'][1:n + 1].astype(int),
        'strength_in': g['str_in'][1:n + 1], 'strength_out': g['str_out'][1:n + 1],
        'strength_total': g['str_total'][1:n + 1], 'cascade_size': cascade[1:n + 1],
    })
    df['cascade_percentage'] = df['cascade_size'] / network_size
    return df

## Sanity: `p = 0` reproduces the baseline, and cascades grow monotonically with `p` (sparse `0_8`, fast)

In [12]:
for p in [0.0, 0.20, 0.40]:
    df = compute_node_cascades(R_DIR / 'network_er_avgdeg_0_8_edges.csv', 0.8, p=p)
    print(f'p={int(p*100):>2d}%  Ki={(1-p)*KI_BASE:.4f}  mean cascade={df.cascade_size.mean():.3f}  max={df.cascade_size.max()}')

p= 0%  Ki=0.0400  mean cascade=8.234  max=293
p=20%  Ki=0.0320  mean cascade=10.018  max=334
p=40%  Ki=0.0240  mean cascade=10.018  max=334


## p-threshold sweep on the target network and save (target schema = Dataset 1/2)

In [13]:
P_THRESHOLDS = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]   # p0 baseline + p5..p40
EDGES = R_DIR / f'network_er_avgdeg_{AVG_DEGREE_LABEL}_edges.csv'
OUT_DIR = R_DIR

RUN_P_SWEEP = True   # heavy on the dense 2_7 network; set True to generate every p-threshold file
if RUN_P_SWEEP:
    summary = []
    for p in P_THRESHOLDS:
        p_label = f'p{int(round(p * 100))}'
        df = compute_node_cascades(EDGES, avg_degree=AVG_DEGREE, p=p)
        df.to_csv(OUT_DIR / f'network_er_avgdeg_{AVG_DEGREE_LABEL}_nodes_with_cascade_{p_label}.csv', index=False)
        target = (df[['node_id', 'cascade_size']]
                  .rename(columns={'node_id': 'bank_id', 'cascade_size': 'systemic_risk_label'}))
        target['log_systemic_risk_label'] = np.log1p(target['systemic_risk_label'])
        target.to_csv(OUT_DIR / f'network_er_avgdeg_{AVG_DEGREE_LABEL}_target_{p_label}.csv', index=False)
        summary.append({'p': p_label, 'Ki': round((1 - p) * KI_BASE, 4),
                        'cascade_mean': round(df.cascade_size.mean(), 1), 'cascade_max': int(df.cascade_size.max())})
        print(summary[-1])
    display(pd.DataFrame(summary))
else:
    print('RUN_P_SWEEP = False - set True to generate p0..p40 node + target files.')

{'p': 'p0', 'Ki': 0.04, 'cascade_mean': np.float64(3433.5), 'cascade_max': 6275}
{'p': 'p5', 'Ki': 0.038, 'cascade_mean': np.float64(3825.9), 'cascade_max': 6322}
{'p': 'p10', 'Ki': 0.036, 'cascade_mean': np.float64(3825.9), 'cascade_max': 6322}
{'p': 'p15', 'Ki': 0.034, 'cascade_mean': np.float64(3825.9), 'cascade_max': 6322}
{'p': 'p20', 'Ki': 0.032, 'cascade_mean': np.float64(3964.9), 'cascade_max': 6326}
{'p': 'p25', 'Ki': 0.03, 'cascade_mean': np.float64(3964.9), 'cascade_max': 6326}
{'p': 'p30', 'Ki': 0.028, 'cascade_mean': np.float64(3991.9), 'cascade_max': 6326}
{'p': 'p35', 'Ki': 0.026, 'cascade_mean': np.float64(3991.9), 'cascade_max': 6326}
{'p': 'p40', 'Ki': 0.024, 'cascade_mean': np.float64(3994.5), 'cascade_max': 6326}


,p,Ki,cascade_mean,cascade_max
0,p0,0.040,3433.5,6275
1,p5,0.038,3825.9,6322
2,p10,0.036,3825.9,6322
3,p15,0.034,3825.9,6322
4,p20,0.032,3964.9,6326
5,p25,0.030,3964.9,6326
6,p30,0.028,3991.9,6326
7,p35,0.026,3991.9,6326
8,p40,0.024,3994.5,6326
